## Clone the repo + enter the dir

In [ ]:
# Go back to root
%cd /content


# Clone fresh
!git clone https://github.com/zohir22s/cifar10-ann-cnn-comparison.git

# Enter the directory
%cd cifar10-ann-cnn-comparison

# Create the missing __init__.py files
!touch data/__init__.py
!touch models/__init__.py
!touch training/__init__.py
!touch evaluation/__init__.py
# Test imports
from data.load_data import train_loader, test_loader
from models.ann_model import ANN
print("✅ All imports successful!")

Install req (Optional)

In [ ]:
# !pip install -r requirements.txt

## Imports

In [ ]:
import torch
import numpy as np
import pandas as pd



## Training

In [ ]:
%run training/train_ann.py --epoch=20


In [ ]:
#%run training/train_cnn.py

In [ ]:
#%run training/train_advanced.py

## Load Trained Models


### ANN:

In [ ]:
from models.ann_model import ANN
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ann_model = ANN.from_pretrained("/content/cifar10-ann-cnn-comparison/results/models/ann.pth", device)

### CNN:

In [ ]:
#from models.cnn_model import CNN


### Advanced CNN:

In [ ]:
#from models.cnn_advanced import CNN_advanced

## Collect Predictions

### ANN:

In [ ]:
from evaluation.metrics import accuracy, precision, recall, f1_score
from evaluation.confusion_matrix import generate_confusion_matrix, plot_confusion_matrix
from models.ann_model import ANN
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = ann_model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)

## Compute metrics

In [ ]:
acc = accuracy(y_true, y_pred)
prec = precision(y_true, y_pred)
rec = recall(y_true, y_pred)
f1 = f1_score(y_true, y_pred)


results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [acc, prec, rec, f1]
})
results.to_csv("/content/cifar10-ann-cnn-comparison/results/logs/ann_metrics.csv", index=False)
results

### Confusion Matrix

In [ ]:
cm = generate_confusion_matrix(ann_model, device)
plot_confusion_matrix(cm, save_path='/content/cifar10-ann-cnn-comparison/results/plots/ann_confusion_matrix.png')